# Clasificación de residuos: EfficientNetB3 a mayor resolución + CutMix + ensamble

Continúa el notebook *MobileNetV2_Entrenamiento_Mejorado*. **Ejecutar en Colab con GPU** (Entorno de ejecución → Cambiar tipo → GPU) y luego *Ejecutar todo*. Tarda entre 1 y 1,5 h con una T4.

**Qué hace**
1. Entrena **EfficientNetB3** a 300 px con **CutMix**, fine-tuning más profundo y parada temprana con más paciencia.
2. Usa **exactamente la misma partición 70/15/15** (misma semilla) que el entrenamiento de MobileNetV2, así que la prueba es comparable.
3. Carga el **MobileNetV2 ya entrenado** desde Drive y arma un **ensamble** (promedio ponderado con TTA). El peso se elige **con validación**; la prueba se evalúa **una sola vez**.
4. Muestra los **errores** del ensamble para revisar etiquetas dudosas.

Antes: ejecuta el notebook de MobileNetV2 para que exista `mobilenetv2_mejorado.keras` en Drive (ruta en `CONFIG["ruta_mobilenet"]`).

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NO HAY GPU: activa una en Entorno de ejecución")

## 1. Datos: montar Drive y descomprimir

In [ ]:
import os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

ZIP_DRIVE = "/content/drive/MyDrive/garbage-dataset.zip"   # ajusta si lo moviste
DATA_ROOT = "/content/garbage"
if not os.path.exists(DATA_ROOT):
    os.makedirs(DATA_ROOT)
    shutil.copy(ZIP_DRIVE, "/content/garbage-dataset.zip")
    with zipfile.ZipFile("/content/garbage-dataset.zip") as z:
        z.extractall(DATA_ROOT)

DATA_DIR = os.path.join(DATA_ROOT, "garbage-dataset")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = DATA_ROOT
print(DATA_DIR, os.listdir(DATA_DIR))

In [ ]:
# Tamaños reales de las imágenes (referencia)
import glob, collections
from PIL import Image
tam = collections.Counter()
for f in glob.glob(DATA_DIR + "/*/*")[:1500]:
    try:
        with Image.open(f) as im: tam[im.size] += 1
    except Exception: pass
print(tam.most_common(8))

## 2. Configuración
`img_size` y `conservar_proporcion` son los dos interruptores a probar. Si la GPU se queda sin memoria, baja `batch` a 16.

In [ ]:
CONFIG = dict(
    backbone="efficientnetb3",
    img_size=300,                # tamaño nativo de EfficientNetB3 (prueba 380 con efficientnetb4)
    conservar_proporcion=False,  # True: resize con relleno (no deforma las fotos 4:3)
    batch=32,
    epochs_head=8,
    epochs_ft=25,
    lr_head=1e-3,
    lr_ft=3e-5,
    ft_layers=120,               # capas finales de la base que se descongelan
    dense=256, dropout1=0.3, dropout2=0.2,
    label_smoothing=0.1,
    cutmix_prob=0.5,             # 0 para desactivar CutMix
    cv_folds=0,
    mixed_precision=True,
    seed=42,                     # misma semilla que MobileNetV2 -> misma partición
    out_dir="/content/salida_eff",
    drive_out="/content/drive/MyDrive/modelos_upa/eff300_ensamble",
    ruta_mobilenet="/content/drive/MyDrive/modelos_upa/entrenamiento_mejorado/mobilenetv2_mejorado.keras",
)
CLASES = ["battery", "biological", "cardboard", "glass", "metal", "paper", "plastic", "trash"]

In [ ]:
import glob, hashlib, json, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
from tensorflow.keras import callbacks, layers, losses, models, optimizers

C = CONFIG
IMG = C["img_size"]
AUTOTUNE = tf.data.AUTOTUNE
OUT = C["out_dir"]
os.makedirs(OUT, exist_ok=True)
tf.keras.utils.set_random_seed(C["seed"])
if gpus and C["mixed_precision"]:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Política de precisión:", tf.keras.mixed_precision.global_policy().name)

## 3. Limpieza y partición 70/15/15 (idéntica a la de MobileNetV2)

In [ ]:
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".gif")
rutas, y = [], []
ilegibles = duplicados = 0
vistos = set()
for i, c in enumerate(CLASES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if not f.lower().endswith(EXT):
            ilegibles += 1
            continue
        try:
            with Image.open(f) as im:
                im.verify()
            with open(f, "rb") as fh:
                h = hashlib.md5(fh.read()).hexdigest()
        except Exception:
            ilegibles += 1
            continue
        if h in vistos:
            duplicados += 1
            continue
        vistos.add(h)
        rutas.append(f)
        y.append(i)
rutas, y = np.array(rutas), np.array(y)
print(f"Imágenes válidas: {len(rutas)} | ilegibles/formato no soportado: {ilegibles} | duplicados exactos: {duplicados}")

r_tr, r_tmp, y_tr, y_tmp = train_test_split(rutas, y, test_size=0.30, stratify=y, random_state=C["seed"])
r_va, r_te, y_va, y_te = train_test_split(r_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=C["seed"])
for nombre, yy in (("entrenamiento", y_tr), ("validación", y_va), ("prueba", y_te)):
    print(nombre, len(yy), np.bincount(yy, minlength=len(CLASES)).tolist())
json.dump({"class_names": CLASES}, open(f"{OUT}/clases.json", "w"))
split_info = {"validas": int(len(rutas)), "ilegibles": int(ilegibles), "duplicados": int(duplicados),
              "train": int(len(y_tr)), "val": int(len(y_va)), "test": int(len(y_te))}

## 4. Pipeline tf.data + CutMix

In [ ]:
def cargar(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    if C["conservar_proporcion"]:
        img = tf.image.resize_with_pad(img, IMG, IMG)
    else:
        img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(tf.round(img), tf.uint8), label        # uint8: la caché ocupa 4 veces menos


def dataset(paths, labels, train=False, cache=True, onehot=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if train:
        ds = ds.shuffle(2048, seed=C["seed"])
    n = len(CLASES)
    ds = ds.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n) if onehot else l),
                num_parallel_calls=AUTOTUNE)
    return ds.batch(C["batch"]).prefetch(AUTOTUNE)


ds_tr = dataset(r_tr, y_tr, train=True)
ds_va = dataset(r_va, y_va)
ds_te = dataset(r_te, y_te)
pesos = dict(enumerate(class_weight.compute_class_weight("balanced", classes=np.arange(len(CLASES)), y=y_tr)))
print({CLASES[k]: round(float(v), 3) for k, v in pesos.items()})

In [ ]:
def _beta(a):
    g1, g2 = tf.random.gamma([], a), tf.random.gamma([], a)
    return g1 / (g1 + g2)


def cutmix(x, y):
    """Pega un recorte de otra imagen del lote y mezcla las etiquetas por área."""
    def aplicar():
        lam = _beta(1.0)
        h, w = tf.shape(x)[1], tf.shape(x)[2]
        rh = tf.cast(tf.cast(h, tf.float32) * tf.sqrt(1.0 - lam), tf.int32)
        rw = tf.cast(tf.cast(w, tf.float32) * tf.sqrt(1.0 - lam), tf.int32)
        cy = tf.random.uniform([], 0, h, dtype=tf.int32)
        cx = tf.random.uniform([], 0, w, dtype=tf.int32)
        y1, y2 = tf.clip_by_value(cy - rh // 2, 0, h), tf.clip_by_value(cy + rh // 2, 0, h)
        x1, x2 = tf.clip_by_value(cx - rw // 2, 0, w), tf.clip_by_value(cx + rw // 2, 0, w)
        idx = tf.random.shuffle(tf.range(tf.shape(x)[0]))
        mascara = tf.pad(tf.ones([y2 - y1, x2 - x1]), [[y1, h - y2], [x1, w - x2]])[None, :, :, None]
        xn = x * (1.0 - mascara) + tf.gather(x, idx) * mascara
        area = tf.cast((y2 - y1) * (x2 - x1), tf.float32) / tf.cast(h * w, tf.float32)
        return xn, y * (1.0 - area) + tf.gather(y, idx) * area
    return tf.cond(tf.random.uniform([]) < C["cutmix_prob"], aplicar, lambda: (x, y))


if C["cutmix_prob"] > 0:
    ds_tr = ds_tr.map(cutmix, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
print("CutMix:", C["cutmix_prob"], "| imagen:", IMG, "| proporción conservada:", C["conservar_proporcion"])

## 5. Modelo

In [ ]:
def get_backbone(nombre):
    s = (IMG, IMG, 3)
    if nombre == "mobilenetv2":
        return tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=s), True
    if nombre == "efficientnetb3":
        return tf.keras.applications.EfficientNetB3(include_top=False, weights="imagenet", input_shape=s), False
    if nombre == "efficientnetv2b0":
        return tf.keras.applications.EfficientNetV2B0(include_top=False, weights="imagenet", input_shape=s), False
    raise ValueError(nombre)
# El segundo valor indica si hay que reescalar a [-1, 1] (MobileNetV2). Las EfficientNet ya normalizan por dentro (entrada 0-255).


def construir():
    base, reescalar = get_backbone(C["backbone"])
    base.trainable = False
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.06), layers.RandomZoom(0.2),
        layers.RandomTranslation(0.1, 0.1), layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2, value_range=(0, 255))], name="aumento")   # solo activo al entrenar
    inp = layers.Input((IMG, IMG, 3))
    x = aug(inp)
    if reescalar:
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)            # preprocesamiento oficial de MobileNetV2
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(C["dropout1"])(x)
    x = layers.Dense(C["dense"], activation="relu")(x)
    x = layers.Dropout(C["dropout2"])(x)
    out = layers.Dense(len(CLASES), activation="softmax", dtype="float32")(x)
    return models.Model(inp, out, name=f"{C['backbone']}_residuos"), base


def compilar(m, lr):
    m.compile(optimizer=optimizers.Adam(lr),
              loss=losses.CategoricalCrossentropy(label_smoothing=C["label_smoothing"]),
              metrics=["accuracy"])


modelo, base = construir()
compilar(modelo, C["lr_head"])
modelo.summary()

In [ ]:
cv_res = None   # sin validación cruzada en este notebook

## 6. Entrenamiento fase 1: cabeza con la base congelada

In [ ]:
t0 = time.time()
cb1 = [callbacks.ModelCheckpoint(f"{OUT}/mejor_fase1.keras", monitor="val_accuracy", save_best_only=True),
       callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
       callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)]
h1 = modelo.fit(ds_tr, validation_data=ds_va, epochs=C["epochs_head"], callbacks=cb1, class_weight=pesos)

## 7. Fase 2: fine-tuning más profundo (BatchNorm congelada, tasa baja)

In [ ]:
base.trainable = True
for capa in base.layers[:-C["ft_layers"]]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False
compilar(modelo, C["lr_ft"])          # recompilar tras cambiar trainable

ini = len(h1.history["loss"])
cb2 = [callbacks.ModelCheckpoint(f"{OUT}/mejor_final.keras", monitor="val_accuracy", save_best_only=True),
       callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
       callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]
h2 = modelo.fit(ds_tr, validation_data=ds_va, epochs=ini + C["epochs_ft"], initial_epoch=ini,
                callbacks=cb2, class_weight=pesos)
t_train = time.time() - t0
print(f"Tiempo total de entrenamiento: {t_train / 60:.1f} min")

In [ ]:
# copia de seguridad en Drive (por si Colab se desconecta)
os.makedirs(C["drive_out"], exist_ok=True)
for f in ("mejor_fase1.keras", "mejor_final.keras"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", C["drive_out"])
print(os.listdir(C["drive_out"]))

## 8. Elegir el mejor entre fase 1 y fase 2 (por validación)

In [ ]:
cands = {}
for nombre, ruta in (("fase 1 (cabeza)", f"{OUT}/mejor_fase1.keras"), ("fase 2 (fine-tuning)", f"{OUT}/mejor_final.keras")):
    if os.path.exists(ruta):
        m = tf.keras.models.load_model(ruta)
        loss, acc = m.evaluate(ds_va, verbose=0)
        cands[nombre] = (acc, loss, m)
        print(f"{nombre}: val_accuracy {acc:.4f} | val_loss {loss:.4f}")
elegido = max(cands, key=lambda k: cands[k][0])
mejor = cands[elegido][2]
print("Modelo elegido:", elegido)

# curvas de entrenamiento
hist = {k: h1.history[k] + h2.history[k] for k in ("accuracy", "val_accuracy", "loss", "val_loss")}
ep = np.arange(1, len(hist["loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (k1, k2, tt) in zip(ax, (("accuracy", "val_accuracy", "Exactitud"), ("loss", "val_loss", "Pérdida"))):
    a.plot(ep, hist[k1], label="Entrenamiento"); a.plot(ep, hist[k2], label="Validación")
    a.axvline(ini + 0.5, color="gray", ls="--", lw=1); a.set_title(tt); a.set_xlabel("Época"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/curvas_entrenamiento.png", dpi=200); plt.show()

## 9. Evaluación de EfficientNetB3 en prueba (con TTA)

In [ ]:
imgs_te = ds_te.map(lambda x, l: x)
p = mejor.predict(imgs_te, verbose=0)
p_flip = mejor.predict(imgs_te.map(lambda x: tf.image.flip_left_right(x)), verbose=0)
ys = np.concatenate([l.numpy().argmax(1) for _, l in ds_te])
y_pred, y_tta = p.argmax(1), ((p + p_flip) / 2).argmax(1)
acc, acc_tta = float((y_pred == ys).mean()), float((y_tta == ys).mean())
print(f"Accuracy en prueba: {acc:.4f} | con TTA: {acc_tta:.4f}")

reporte = classification_report(ys, y_tta, target_names=CLASES, digits=3, output_dict=True)
print(classification_report(ys, y_tta, target_names=CLASES, digits=3))
cm = confusion_matrix(ys, y_tta)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=ax[0])
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=ax[1])
for a, tt in zip(ax, ("Matriz de confusión (conteos)", "Matriz de confusión (por clase real)")):
    a.set_title(tt); a.set_xlabel("Predicción"); a.set_ylabel("Etiqueta real")
plt.tight_layout(); plt.savefig(f"{OUT}/matriz_confusion.png", dpi=200); plt.show()

# latencia por imagen (1 imagen, este equipo)
uno = tf.zeros((1, IMG, IMG, 3))
for _ in range(3):
    mejor.predict(uno, verbose=0)
t = time.time()
for _ in range(20):
    mejor.predict(uno, verbose=0)
lat_ms = (time.time() - t) / 20 * 1000
print(f"Latencia por imagen: {lat_ms:.0f} ms")

## 10. Guardar EfficientNetB3

In [ ]:
nombre_modelo = f"{C['backbone']}_mejorado.keras"
mejor.save(f"{OUT}/{nombre_modelo}")                       # entrada 0-255; el preprocesamiento va dentro

resumen = {
    "backbone": C["backbone"], "modelo_elegido": elegido, "configuracion": C, "particion": split_info,
    "val_accuracy_fase1": float(cands.get("fase 1 (cabeza)", (np.nan,))[0]),
    "val_accuracy_fase2": float(cands.get("fase 2 (fine-tuning)", (np.nan,))[0]),
    "test_accuracy": acc, "test_accuracy_tta": acc_tta,
    "test_macroF1_tta": float(f1_score(ys, y_tta, average="macro")),
    "test_F1_clase_mas_debil": float(min(reporte[c]["f1-score"] for c in CLASES)),
    "validacion_cruzada": cv_res, "latencia_ms": lat_ms, "minutos_entrenamiento": t_train / 60,
    "parametros": int(mejor.count_params()), "reporte": reporte,
}
json.dump(resumen, open(f"{OUT}/metricas_test.json", "w"), indent=2, ensure_ascii=False)

os.makedirs(C["drive_out"], exist_ok=True)
for f in os.listdir(OUT):
    if f.startswith("mejor_fase1") or f.startswith("mejor_final"):
        continue
    shutil.copy(os.path.join(OUT, f), C["drive_out"])
print("Guardado en Drive:", C["drive_out"], os.listdir(C["drive_out"]))

## 11. Ensamble con MobileNetV2 (cada modelo con su tamaño de entrada)
El peso del ensamble se elige **en validación**; la prueba se evalúa **una vez**.

In [ ]:
mob = tf.keras.models.load_model(C["ruta_mobilenet"])
print("MobileNetV2 cargado. Entrada:", mob.input_shape, "| EfficientNet:", mejor.input_shape)


def _cargar(path, size, pad):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize_with_pad(img, size, size) if pad else tf.image.resize(img, (size, size))
    return tf.cast(tf.round(img), tf.float32)


def probs_tta(m, paths, size, pad=False):
    ds = tf.data.Dataset.from_tensor_slices(paths).map(lambda p: _cargar(p, size, pad), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(C["batch"]).prefetch(AUTOTUNE)
    p = m.predict(ds, verbose=0)
    pf = m.predict(ds.map(tf.image.flip_left_right), verbose=0)
    return (p + pf) / 2


S_MOB, S_EFF = mob.input_shape[1], mejor.input_shape[1]
PAD = C["conservar_proporcion"]      # solo EfficientNet se entrenó con esa opción; MobileNetV2 usó resize directo
pv_mob, pv_eff = probs_tta(mob, r_va, S_MOB), probs_tta(mejor, r_va, S_EFF, PAD)
pt_mob, pt_eff = probs_tta(mob, r_te, S_MOB), probs_tta(mejor, r_te, S_EFF, PAD)

# peso elegido con VALIDACIÓN
res_val = []
for w in np.round(np.arange(0, 1.01, 0.1), 1):
    res_val.append((float((((w * pv_eff + (1 - w) * pv_mob).argmax(1)) == y_va).mean()), w))
print("Validación por peso de EfficientNet:", [(w, round(a, 4)) for a, w in res_val])
w_mejor = max(res_val, key=lambda t: (t[0], -abs(t[1] - 0.5)))[1]
print("Peso elegido (EfficientNet):", w_mejor)

# PRUEBA, una sola vez
pt_ens = w_mejor * pt_eff + (1 - w_mejor) * pt_mob
y_ens = pt_ens.argmax(1)
acc_mob, acc_eff, acc_ens = [float((p.argmax(1) == y_te).mean()) for p in (pt_mob, pt_eff, pt_ens)]
print(f"PRUEBA  MobileNetV2 = {acc_mob:.4f} | EfficientNetB3 = {acc_eff:.4f} | ENSAMBLE = {acc_ens:.4f}")
rep_ens = classification_report(y_te, y_ens, target_names=CLASES, digits=3, output_dict=True)
print(classification_report(y_te, y_ens, target_names=CLASES, digits=3))
f1_ens = float(f1_score(y_te, y_ens, average="macro"))
f1_min = float(min(rep_ens[c]["f1-score"] for c in CLASES))
print(f"macro-F1 = {f1_ens:.4f} | F1 mínimo por clase = {f1_min:.4f}")

cm2 = confusion_matrix(y_te, y_ens)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=ax[0])
sns.heatmap(cm2 / cm2.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=ax[1])
for a, tt in zip(ax, ("Ensamble: conteos", "Ensamble: por clase real")):
    a.set_title(tt); a.set_xlabel("Predicción"); a.set_ylabel("Etiqueta real")
plt.tight_layout(); plt.savefig(f"{OUT}/matriz_confusion_ensamble.png", dpi=200); plt.show()

json.dump({"peso_efficientnet": float(w_mejor), "val_por_peso": [(float(w), a) for a, w in res_val],
           "test_accuracy_mobilenet": acc_mob, "test_accuracy_efficientnet": acc_eff, "test_accuracy_ensamble": acc_ens,
           "test_macroF1_ensamble": f1_ens, "test_F1_minimo_ensamble": f1_min, "img_size_efficientnet": int(S_EFF),
           "reporte": rep_ens}, open(f"{OUT}/metricas_ensamble.json", "w"), indent=2, ensure_ascii=False)

## 12. Errores del ensamble (para revisar etiquetas dudosas)

In [ ]:
err = np.where(y_ens != y_te)[0]
print(f"{len(err)} imágenes mal clasificadas de {len(y_te)}")
conf = np.sort(pt_ens[err].max(1))[::-1]
orden = err[np.argsort(-pt_ens[err].max(1))][:24]       # los errores más seguros del modelo: suelen ser etiquetas dudosas
fig, axs = plt.subplots(4, 6, figsize=(18, 12.5))
for a, i in zip(axs.ravel(), orden):
    a.imshow(Image.open(r_te[i]).convert("RGB"))
    a.set_title(f"real: {CLASES[y_te[i]]}\npred: {CLASES[y_ens[i]]} ({pt_ens[i].max():.0%})", fontsize=10)
    a.axis("off")
for a in axs.ravel()[len(orden):]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/errores_ensamble.png", dpi=150); plt.show()
print("Pares de confusión más frecuentes:")
pares = sorted(((cm2[i, j], CLASES[i], CLASES[j]) for i in range(8) for j in range(8) if i != j), reverse=True)[:6]
for n, a, b in pares: print(f"  {a} -> {b}: {n}")

## 13. Guardar en Drive y resumen para el informe

In [ ]:
for f in ("metricas_ensamble.json", "matriz_confusion_ensamble.png", "errores_ensamble.png"):
    shutil.copy(f"{OUT}/{f}", C["drive_out"])
print("Guardado en Drive:", C["drive_out"], os.listdir(C["drive_out"]))
print(f"EfficientNetB3 a {S_EFF}px | CutMix={C['cutmix_prob']} | modelo elegido: {elegido}")
print(f"PRUEBA  MobileNetV2 = {acc_mob:.4f} | EfficientNetB3 = {acc_eff:.4f} | ENSAMBLE (w_eff={w_mejor}) = {acc_ens:.4f}")
print(f"Ensamble: macro-F1 = {f1_ens:.4f} | F1 mínimo = {f1_min:.4f}")